# Selenium을 이용한 기상청 날씨 크롤링

In [1]:
%pip install -q selenium

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.3.2 -> 25.0.1
[notice] To update, run: c:\Users\dandycode\.pyenv\pyenv-win\versions\3.11.7\python.exe -m pip install --upgrade pip


In [1]:
# 봇 처럼 여겨지지 않기 위해 주피터 노트북 ipynb 파일 생성
# 크롤링은 어떻게 사이트에서 사람이 하는 것처럼 보일까가 중요

# pip install selenium
from selenium import webdriver

driver = webdriver.Chrome()
# driver.set_window_size(1920, 1080)
driver.set_window_size(1280, 720)

# URL='https://www.naver.com/'
URL='https://www.weather.go.kr/w/weather/forecast/short-term.do'
driver.get(url=URL)

In [2]:
from selenium.webdriver.common.by import By  # 웹 요소(element)를 찾기 위한 방법을 제공하는 클래스입니다. (예: ID, class, CSS 선택자 등)
from selenium.webdriver.support.ui import WebDriverWait # 웹 페이지에서 특정 조건이 충족될 때까지 기다리게 하는 클래스입니다.
from selenium.webdriver.support import expected_conditions as EC # WebDriverWait와 함께 사용되며, 기다릴 "특정 조건"들을 정의하는 모듈입니다. (예: 요소가 보일 때까지, 클릭 가능할 때까지)

In [3]:
# --- 방법 1: 링크 텍스트 사용 (가장 간단하고 추천) ---
# "1시간 간격"이라는 텍스트를 가진 링크를 직접 찾습니다.
print("방법 1: 링크 텍스트로 클릭 시도...")
# WebDriverWait를 사용하여 요소가 클릭 가능할 때까지 최대 10초간 기다립니다.
one_hour_button = WebDriverWait(driver, 10).until(
    EC.element_to_be_clickable((By.LINK_TEXT, "1시간 간격"))
)
one_hour_button.click()
print("'1시간 간격' 버튼 클릭 성공 (링크 텍스트 사용)")

방법 1: 링크 텍스트로 클릭 시도...
'1시간 간격' 버튼 클릭 성공 (링크 텍스트 사용)


In [4]:
# --- 방법 2: CSS 선택자 사용 ---
print("방법 2: CSS 선택자(클래스)로 클릭 시도...")
table_view_button = WebDriverWait(driver, 10).until(
    EC.element_to_be_clickable((By.CSS_SELECTOR, "a.view-table"))
)
table_view_button.click()
print("'표 형태' 버튼 클릭 성공 (CSS 선택자 - 클래스 사용)")

방법 2: CSS 선택자(클래스)로 클릭 시도...
'표 형태' 버튼 클릭 성공 (CSS 선택자 - 클래스 사용)


In [6]:
from selenium.common.exceptions import NoSuchElementException # Selenium을 사용하여 웹 요소를 찾을 때, 해당 요소를 찾지 못했을 경우 발생하는 예외(오류)를 처리하기 위한 클래스입니다.
import re # 정규표현식 사용 (데이터 정제용)

### 예제1) 날씨 데이터 전처리 로직(직접 작성)

In [7]:
# --- 데이터 저장을 위한 빈 리스트 초기화 ---
times = []
weathers = []
temperatures = []
feels_like_temps = []
precip_amounts = []
precip_intensities = []
precip_probabilities = []
wind_directions = []
wind_speeds = []
humidities = []
heatwave_impacts = []

# --- 데이터 추출 로직 ---
try:
    # 데이터 항목들을 포함하는 부모 div 찾기
    # daily_div = driver.find_element(By.CSS_SELECTOR, "div.daily")
    # item_wrap = daily_div.find_element(By.CSS_SELECTOR, "div.item-wrap")
    # 위 코드를 > 를 이용해 한줄로 작성 가능
    item_wrap = driver.find_element(By.CSS_SELECTOR, "div.daily > div.item-wrap")

    # print(item_wrap.get_attribute('outerHTML')) # item_wrap 내용 확인

    # 각 시간대별 데이터 묶음 (ul 태그) 찾기
    item_list = item_wrap.find_elements(By.CSS_SELECTOR, "ul.item")

    print(f"총 {len(item_list)}개의 시간대 데이터를 찾았습니다.")

    # 각 시간대별로 반복 처리
    for item_ul in item_list:
        # 각 ul 내의 li 요소들을 리스트로 가져오기
        # IndexError를 방지하기 위해 li 개수를 먼저 확인하는 것이 더 안전할 수 있습니다.
        try:
             li_elements = item_ul.find_elements(By.TAG_NAME, "li")
             # 최소 필요한 li 개수(예: 10개) 확인 로직 추가 가능
             # if len(li_elements) < 10: continue # 또는 None 추가 후 다음 item으로
        except NoSuchElementException:
             print("경고: 현재 시간대(ul.item)에서 li 요소들을 찾을 수 없습니다. 건너<0xEB><0x8D>니다.")
             # 모든 리스트에 None 추가하고 다음 item_ul로 넘어감
             lists_to_update = [times, weathers, temperatures, feels_like_temps, precip_amounts, precip_intensities, precip_probabilities, wind_directions, wind_speeds, humidities, heatwave_impacts]
             for lst in lists_to_update:
                 lst.append(None)
             continue # 다음 시간대로

        # 각 항목 추출 및 정제 (clean_value 함수 로직 인라인)

        # 1. 시각
        cleaned_time = None
        try:
            time_text = li_elements[0].find_element(By.CSS_SELECTOR, "span:not(.hid)").text
            # 정제 로직 (기본 전처리)
            time_text = time_text.strip().replace('&nbsp;', '')
            if time_text and time_text != '-':
                cleaned_time = time_text # 시각은 특별한 숫자 변환 없음
        except (NoSuchElementException, IndexError) as e:
             print(f"시각 처리 오류: {e}") # 디버깅용 로그
        times.append(cleaned_time)


        # 2. 날씨
        cleaned_weather = None
        try:
            # 먼저 wic 클래스 시도
            try:
                 weather_text = li_elements[1].find_element(By.CSS_SELECTOR, "span.wic").text
            except NoSuchElementException:
                 # wic 없으면 다른 span 시도
                 weather_text = li_elements[1].find_element(By.CSS_SELECTOR, "span:not(.hid)").text
            
            # 정제 로직 (기본 전처리)
            weather_text = weather_text.strip().replace('&nbsp;', '')
            if weather_text and weather_text != '-':
                 cleaned_weather = weather_text # 날씨는 텍스트 그대로
        except (NoSuchElementException, IndexError) as e:
            print(f"날씨 처리 오류: {e}")
        weathers.append(cleaned_weather)


        # 3. 기온 
        # 참고: 원래 코드에서는 li_elements[2] (3번째 li)에서 추출했으나, 
        # 이전 논의에서 실제 기온은 4번째 li에서 가져오는 것이 맞다고 판단했습니다.
        # 만약 3번째 li의 텍스트에서 첫 숫자를 기온으로 사용하려면 아래 로직 사용
        cleaned_temp = None
        try:
             # 3번째 li의 전체 텍스트 (예: "16℃(16℃)") 에서 첫 숫자 추출
             temp_text_combined = li_elements[2].find_element(By.CSS_SELECTOR, "span.hid.feel").text
             temp_text_combined = temp_text_combined.strip().replace('&nbsp;', '')
             if temp_text_combined and temp_text_combined != '-':
                 match = re.search(r'-?\d+', temp_text_combined) # 첫 번째 숫자 검색
                 if match:
                     cleaned_temp = int(match.group(0))
        except (NoSuchElementException, IndexError, ValueError) as e:
            print(f"기온 처리 오류: {e}")
        temperatures.append(cleaned_temp)
        

        # 4. 기온 체감 (3번째 li의 span.chill 텍스트)
        cleaned_feels_like = None
        try:
            chill_text = li_elements[2].find_element(By.CSS_SELECTOR, "span.chill").text # 예: (16℃)
            chill_text = chill_text.strip().replace('&nbsp;', '')
            if chill_text and chill_text != '-':
                match = re.search(r'-?\d+', chill_text) # 괄호 안 숫자 검색
                if match:
                    cleaned_feels_like = int(match.group(0))
        except (NoSuchElementException, IndexError, ValueError) as e:
            print(f"체감기온 처리 오류: {e}")
        feels_like_temps.append(cleaned_feels_like)


        # 5. 강수량
        cleaned_precip_amount = None
        try:
            pcp_text = li_elements[4].find_element(By.CSS_SELECTOR, "span:not(.hid)").text
            pcp_text = pcp_text.strip().replace('&nbsp;', '')
            if pcp_text and pcp_text != '-':
                if '빗방울' in pcp_text:
                    cleaned_precip_amount = 0.0 # '빗방울'은 0.0으로 처리
                else:
                    match = re.search(r'\d+\.?\d*', pcp_text) # 소수점 포함 숫자 검색
                    if match:
                        cleaned_precip_amount = float(match.group(0))
        except (NoSuchElementException, IndexError, ValueError) as e:
            print(f"강수량 처리 오류: {e}")
        precip_amounts.append(cleaned_precip_amount)


        # 6. 강수강도
        cleaned_intensity = None
        try:
            intensity_element = li_elements[5]
            intensity_text = None
            # 먼저 span 찾기 시도
            try:
                intensity_text = intensity_element.find_element(By.CSS_SELECTOR, "span:not(.hid)").text
            except NoSuchElementException:
                 # span 없으면 li 전체 텍스트에서 hid 제외
                 full_text = intensity_element.text
                 hidden_text = ""
                 try:
                     hidden_text = intensity_element.find_element(By.CSS_SELECTOR, "span.hid").text
                 except NoSuchElementException: pass
                 intensity_text = full_text.replace(hidden_text, "").strip()

            # 정제 로직 (기본 전처리)
            intensity_text = intensity_text.strip().replace('&nbsp;', '')
            if intensity_text and intensity_text != '-':
                cleaned_intensity = intensity_text # 텍스트 그대로
        except (NoSuchElementException, IndexError) as e:
             print(f"강수강도 처리 오류: {e}")
        precip_intensities.append(cleaned_intensity)


        # 7. 강수확률
        cleaned_prob = None
        try:
            prob_text = li_elements[6].find_element(By.CSS_SELECTOR, "span:not(.hid)").text
            prob_text = prob_text.strip().replace('&nbsp;', '')
            if prob_text and prob_text != '-':
                match = re.search(r'\d+', prob_text) # % 제거 후 숫자만
                if match:
                    cleaned_prob = int(match.group(0))
        except (NoSuchElementException, IndexError, ValueError) as e:
            print(f"강수확률 처리 오류: {e}")
        precip_probabilities.append(cleaned_prob)


        # 8. 바람 (방향, 속도 분리)
        cleaned_wind_dir = None
        cleaned_wind_spd = None
        try:
            wind_li = li_elements[7]
            # 바람 방향
            try:
                wind_dir_text = wind_li.find_element(By.CSS_SELECTOR, "span.wdic").text
                wind_dir_text = wind_dir_text.strip().replace('&nbsp;', '')
                if wind_dir_text and wind_dir_text != '-':
                     cleaned_wind_dir = wind_dir_text
            except NoSuchElementException: pass # 없으면 None 유지
            # 바람 속도
            try:
                wind_spd_text = wind_li.find_element(By.CSS_SELECTOR, "span.wspd:not(.qwsd)").text
                wind_spd_text = wind_spd_text.strip().replace('&nbsp;', '')
                if wind_spd_text and wind_spd_text != '-':
                    match = re.search(r'\d+', wind_spd_text) # m/s 제거 후 숫자만
                    if match:
                        cleaned_wind_spd = int(match.group(0))
            except NoSuchElementException: pass # 없으면 None 유지
        except (NoSuchElementException, IndexError, ValueError) as e:
            print(f"바람 처리 오류: {e}")
        wind_directions.append(cleaned_wind_dir)
        wind_speeds.append(cleaned_wind_spd)


        # 9. 습도
        cleaned_humidity = None
        try:
            hum_text = li_elements[8].find_element(By.CSS_SELECTOR, "span:not(.hid)").text
            hum_text = hum_text.strip().replace('&nbsp;', '')
            if hum_text and hum_text != '-':
                match = re.search(r'\d+', hum_text) # % 제거 후 숫자만
                if match:
                    cleaned_humidity = int(match.group(0))
        except (NoSuchElementException, IndexError, ValueError) as e:
            print(f"습도 처리 오류: {e}")
        humidities.append(cleaned_humidity)


        # 10. 폭염 영향
        cleaned_heatwave = None
        try:
            heat_text = li_elements[9].find_element(By.CSS_SELECTOR, "span:not(.hid)").text
            heat_text = heat_text.strip().replace('&nbsp;', '')
            if heat_text and heat_text != '-':
                cleaned_heatwave = heat_text # 텍스트 그대로
        except (NoSuchElementException, IndexError) as e:
            print(f"폭염영향 처리 오류: {e}")
        heatwave_impacts.append(cleaned_heatwave)

    # --- 최종 결과 출력 ---
    # (이전과 동일)
    print("\n--- 추출 완료된 리스트 ---")
    print(f"시각: {times}")
    print(f"날씨: {weathers}")
    print(f"기온(℃): {temperatures}")
    print(f"체감기온(℃): {feels_like_temps}")
    print(f"강수량(mm): {precip_amounts}")
    print(f"강수강도: {precip_intensities}")
    print(f"강수확률(%): {precip_probabilities}")
    print(f"바람방향: {wind_directions}")
    print(f"바람속도(m/s): {wind_speeds}")
    print(f"습도(%): {humidities}")
    print(f"폭염영향: {heatwave_impacts}")

except NoSuchElementException as e:
    print(f"오류: 필수 요소를 찾을 수 없습니다. CSS 선택자를 확인하세요. ({e})")
except Exception as e:
    print(f"예상치 못한 오류 발생: {e}")
    import traceback
    traceback.print_exc()

# finally:
#     # 작업 완료 후 드라이버 종료
#     # driver.quit()

총 14개의 시간대 데이터를 찾았습니다.

--- 추출 완료된 리스트 ---
시각: ['11시', '12시', '13시', '14시', '15시', '16시', '17시', '18시', '19시', '20시', '21시', '22시', '23시', '0시']
날씨: ['구름 많음', '맑음', '맑음', '흐림', '맑음', '맑음', '맑음', '맑음', '맑음', '맑음', '맑음', '맑음', '맑음', '맑음']
기온(℃): [31, 32, 32, 30, 30, 30, 30, 29, 28, 27, 27, 27, 27, 27]
체감기온(℃): [32, 33, 34, 32, 32, 32, 31, 31, 30, 29, 29, 29, 29, 29]
강수량(mm): [None, None, None, None, None, None, None, None, None, None, None, None, None, None]
강수강도: [None, None, None, None, None, None, None, None, None, None, None, None, None, None]
강수확률(%): [None, None, None, None, None, None, 0, 0, 0, 0, 0, 0, 0, 0]
바람방향: ['남동풍', '남동풍', '남동풍', '남동풍', '남동풍', '남풍', '남풍', '남풍', '남풍', '남풍', '남서풍', '남서풍', '남서풍', '서풍']
바람속도(m/s): [1, 2, 2, 3, 4, 3, 3, 3, 3, 2, 2, 2, 2, 2]
습도(%): [70, 70, 75, 75, 75, 75, 70, 75, 75, 85, 85, 85, 85, 85]
폭염영향: ['경고', '경고', '경고', '경고', '경고', '경고', '경고', '경고', '경고', '경고', '경고', '경고', '경고', '경고']


In [8]:
keys = ['시각', '날씨', '기온(℃)', '체감기온(℃)', '강수량(mm)', '강수강도', '강수확률(%)', '바람방향', '바람속도(m/s)', '습도(%)', '폭염영향']
# 제공된 리스트 변수들을 사용한다고 가정 (times, weathers, temperatures 등)
list_of_lists = [times, weathers, temperatures, feels_like_temps, precip_amounts, precip_intensities, precip_probabilities, wind_directions, wind_speeds, humidities, heatwave_impacts]

structured_data = []
num_items = len(times) # 모든 리스트 길이가 같다고 가정

for i in range(num_items):
    record = {}
    for j, key in enumerate(keys):
         # list_of_lists[j][i] 를 사용하여 올바른 값에 접근
         record[key] = list_of_lists[j][i] 
    structured_data.append(record)

# 이제 structured_data를 JSON으로 변환하여 API에 전달할 수 있습니다.
import json
json_data = json.dumps(structured_data, ensure_ascii=False, indent=2) 
print(type(json_data))
print(json_data)

<class 'str'>
[
  {
    "시각": "11시",
    "날씨": "구름 많음",
    "기온(℃)": 31,
    "체감기온(℃)": 32,
    "강수량(mm)": null,
    "강수강도": null,
    "강수확률(%)": null,
    "바람방향": "남동풍",
    "바람속도(m/s)": 1,
    "습도(%)": 70,
    "폭염영향": "경고"
  },
  {
    "시각": "12시",
    "날씨": "맑음",
    "기온(℃)": 32,
    "체감기온(℃)": 33,
    "강수량(mm)": null,
    "강수강도": null,
    "강수확률(%)": null,
    "바람방향": "남동풍",
    "바람속도(m/s)": 2,
    "습도(%)": 70,
    "폭염영향": "경고"
  },
  {
    "시각": "13시",
    "날씨": "맑음",
    "기온(℃)": 32,
    "체감기온(℃)": 34,
    "강수량(mm)": null,
    "강수강도": null,
    "강수확률(%)": null,
    "바람방향": "남동풍",
    "바람속도(m/s)": 2,
    "습도(%)": 75,
    "폭염영향": "경고"
  },
  {
    "시각": "14시",
    "날씨": "흐림",
    "기온(℃)": 30,
    "체감기온(℃)": 32,
    "강수량(mm)": null,
    "강수강도": null,
    "강수확률(%)": null,
    "바람방향": "남동풍",
    "바람속도(m/s)": 3,
    "습도(%)": 75,
    "폭염영향": "경고"
  },
  {
    "시각": "15시",
    "날씨": "맑음",
    "기온(℃)": 30,
    "체감기온(℃)": 32,
    "강수량(mm)": null,
    "강수강도": null,
    "강수확률(%)": null,
    

In [5]:
driver

<selenium.webdriver.chrome.webdriver.WebDriver (session="9a3d35dc54ecf3f1140a4f31e5b9cb9c")>

### 예제2) Gemini에서 html 파일 전달하여 날씨 데이터 전처리하는 함수 생성한 경우

1. 크롤링 대상 HTML을 복사하여 `temp.html` 파일 생성
2. 생성한 파일을 Gemini에 전달하되, 최종적으로 얻고자 하는 데이터 구조 예시(여기에서는 JSON)를 함께 작성하여 전달

In [6]:
def parse_weather_data(driver):
    """
    WebDriver를 사용하여 웹페이지에서 날씨 정보를 파싱합니다.
    
    Args:
        driver (webdriver.Chrome): 현재 웹페이지가 로드된 WebDriver 인스턴스.
        
    Returns:
        list: 날씨 정보가 담긴 딕셔너리 리스트.
    """
    weather_data_list = []
    
    try:
        # 1. CSS 선택자를 사용하여 날씨 데이터가 포함된 모든 <ul> 요소를 찾습니다.
        items = driver.find_elements(By.CSS_SELECTOR, 'ul.item, ul.vs-item, ul.s-item')
        
        for item in items:
            # 2. 각 <ul> 요소 내에서 세부 데이터를 추출합니다.
            try:
                time_element = item.find_element(By.XPATH, ".//li/span[text()='시각: ']/following-sibling::span")
                time_str = time_element.text.strip()
            except:
                time_str = None
            
            try:
                weather_element = item.find_element(By.CSS_SELECTOR, 'span.wic')
                weather_str = weather_element.get_attribute('title')
            except:
                weather_str = None
            
            try:
                temp_full_element = item.find_element(By.XPATH, ".//li/span[text()='기온(체감온도) ']/following-sibling::span")
                temp_full_str = temp_full_element.text.strip()
                temp_value = temp_full_str.split('℃')[0].strip()
                feel_value = temp_full_str.split('℃')[1].strip().replace('(','').replace(')','')
            except:
                temp_value = None
                feel_value = None
            
            try:
                wind_direction_element = item.find_element(By.CSS_SELECTOR, 'span.wdic')
                wind_direction = wind_direction_element.get_attribute('title')
            except:
                wind_direction = None
            
            try:
                wind_speed_element = item.find_element(By.XPATH, ".//span[@class='wspd'][contains(text(),'m/s')]")
                wind_speed_str = wind_speed_element.text.strip().replace('m/s', '')
            except:
                wind_speed_str = None
                
            try:
                rain_amount_element = item.find_element(By.XPATH, ".//li[@class='pcp']/span[text()='강수량: ']/following-sibling::span")
                rain_amount_str = rain_amount_element.text.strip()
            except:
                rain_amount_str = None

            try:
                rain_intensity_element = item.find_element(By.XPATH, ".//li/span[text()='강수강도: ']/following-sibling::span")
                rain_intensity_str = rain_intensity_element.text.strip()
            except:
                rain_intensity_str = None
            
            try:
                rain_prob_element = item.find_element(By.XPATH, ".//li/span[text()='강수확률: ']/following-sibling::span")
                rain_prob_str = rain_prob_element.text.strip().replace('%','')
            except:
                rain_prob_str = None
            
            try:
                humidity_element = item.find_element(By.XPATH, ".//li/span[text()='습도: ']/following-sibling::span")
                humidity_str = humidity_element.text.strip().replace('%','')
            except:
                humidity_str = None
                
            try:
                heat_impact_element = item.find_element(By.XPATH, ".//li/span[text()='폭염영향: ']/following-sibling::span")
                heat_impact = heat_impact_element.text.strip()
            except:
                heat_impact = None
            
            # 3. 추출된 데이터를 딕셔너리에 저장합니다.
            weather_dict = {
                "시각": time_str,
                "날씨": weather_str,
                "기온(℃)": int(temp_value) if temp_value and temp_value.isdigit() else None,
                "체감기온(℃)": int(feel_value) if feel_value and feel_value.isdigit() else None,
                "강수량(mm)": float(rain_amount_str) if rain_amount_str and rain_amount_str.replace('.', '', 1).isdigit() else None,
                "강수강도": rain_intensity_str if rain_intensity_str and rain_intensity_str != '-' else None,
                "강수확률(%)": int(rain_prob_str) if rain_prob_str and rain_prob_str.isdigit() else None,
                "바람방향": wind_direction,
                "바람속도(m/s)": float(wind_speed_str) if wind_speed_str else None,
                "습도(%)": int(humidity_str) if humidity_str and humidity_str.isdigit() else None,
                "폭염영향": heat_impact
            }
            weather_data_list.append(weather_dict)
            
    except Exception as e:
        print(f"오류 발생: {e}")
        
    return weather_data_list

In [7]:
weather_result = parse_weather_data(driver)

In [8]:
type(weather_result)

list

In [9]:
# 이제 structured_data를 JSON으로 변환하여 API에 전달할 수 있습니다.
import json
json_data = json.dumps(weather_result, ensure_ascii=False, indent=2) 
print(type(json_data))
print(json_data)

<class 'str'>
[
  {
    "시각": "22시",
    "날씨": "맑음",
    "기온(℃)": 28,
    "체감기온(℃)": 30,
    "강수량(mm)": null,
    "강수강도": null,
    "강수확률(%)": null,
    "바람방향": "서풍",
    "바람속도(m/s)": null,
    "습도(%)": 75,
    "폭염영향": "경고"
  },
  {
    "시각": "23시",
    "날씨": "맑음",
    "기온(℃)": 27,
    "체감기온(℃)": 29,
    "강수량(mm)": null,
    "강수강도": null,
    "강수확률(%)": null,
    "바람방향": "서풍",
    "바람속도(m/s)": null,
    "습도(%)": 80,
    "폭염영향": "경고"
  },
  {
    "시각": "0시",
    "날씨": "맑음",
    "기온(℃)": 27,
    "체감기온(℃)": 29,
    "강수량(mm)": null,
    "강수강도": null,
    "강수확률(%)": null,
    "바람방향": "북서풍",
    "바람속도(m/s)": null,
    "습도(%)": 80,
    "폭염영향": "경고"
  },
  {
    "시각": "01시",
    "날씨": "맑음",
    "기온(℃)": null,
    "체감기온(℃)": null,
    "강수량(mm)": null,
    "강수강도": null,
    "강수확률(%)": null,
    "바람방향": "북서풍",
    "바람속도(m/s)": null,
    "습도(%)": null,
    "폭염영향": ""
  },
  {
    "시각": "02시",
    "날씨": "맑음",
    "기온(℃)": null,
    "체감기온(℃)": null,
    "강수량(mm)": null,
    "강수강도": null,
    "강수확률(%

# GEMINI API 연동

In [17]:
import os
from dotenv import load_dotenv
load_dotenv()

gemini_api_key = os.getenv("GEMINI_API_KEY")
if not gemini_api_key:
    raise ValueError("GEMINI_API_KEY 환경 변수를 설정해주세요.")

In [11]:
%pip install -q -U google-genai

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.3.2 -> 25.0.1
[notice] To update, run: c:\Users\dandycode\.pyenv\pyenv-win\versions\3.11.7\python.exe -m pip install --upgrade pip


In [22]:
from google import genai

client = genai.Client(api_key=gemini_api_key)

response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents="Explain how AI works in a few words",
)

print(response.text)

AI learns patterns from data to make decisions or predictions.


In [23]:
prompt = f"""다음은 시간대별 날씨 데이터입니다. 이 JSON 데이터를 분석하여 날씨 상황을 요약하고, 특히 주목할 만한 변화(예: 강수 시작/종료, 풍속 변화 등)를 설명해주세요.

**날씨 데이터:**
```json
{json_data}
```
"""

print(prompt)

다음은 시간대별 날씨 데이터입니다. 이 JSON 데이터를 분석하여 날씨 상황을 요약하고, 특히 주목할 만한 변화(예: 강수 시작/종료, 풍속 변화 등)를 설명해주세요.

**날씨 데이터:**
```json
[
  {
    "시각": "11시",
    "날씨": "구름 많음",
    "기온(℃)": 31,
    "체감기온(℃)": 32,
    "강수량(mm)": null,
    "강수강도": null,
    "강수확률(%)": null,
    "바람방향": "남동풍",
    "바람속도(m/s)": null,
    "습도(%)": 70,
    "폭염영향": "경고"
  },
  {
    "시각": "12시",
    "날씨": "맑음",
    "기온(℃)": 32,
    "체감기온(℃)": 33,
    "강수량(mm)": null,
    "강수강도": null,
    "강수확률(%)": null,
    "바람방향": "남동풍",
    "바람속도(m/s)": null,
    "습도(%)": 70,
    "폭염영향": "경고"
  },
  {
    "시각": "13시",
    "날씨": "맑음",
    "기온(℃)": 32,
    "체감기온(℃)": 34,
    "강수량(mm)": null,
    "강수강도": null,
    "강수확률(%)": null,
    "바람방향": "남동풍",
    "바람속도(m/s)": null,
    "습도(%)": 75,
    "폭염영향": "경고"
  },
  {
    "시각": "14시",
    "날씨": "흐림",
    "기온(℃)": 30,
    "체감기온(℃)": 32,
    "강수량(mm)": null,
    "강수강도": null,
    "강수확률(%)": null,
    "바람방향": "남동풍",
    "바람속도(m/s)": null,
    "습도(%)": 75,
    "폭염영향": "경고"
  },
  {
    "시각": "15시",
   

In [24]:
response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents=prompt,
)
print(response.text)

제공된 시간대별 날씨 데이터를 분석하여 다음과 같이 날씨 상황을 요약하고 주요 변화를 설명합니다.

---

### 날씨 상황 요약 및 주요 변화 분석

제공된 날씨 데이터는 특정 시점부터 다음 날 또는 그 이후까지의 날씨를 시간대별로 보여주고 있습니다. 특히, 11시부터 다음 날 0시까지는 기온, 체감기온, 습도 등의 상세 정보가 제공되며, 그 이후 시간대부터는 대부분의 상세 수치 정보 없이 날씨와 바람 방향 정보 위주로 제공됩니다.

**1. 전반적인 날씨 상황 요약 (11시 - 다음 날 0시)**

*   **폭염 지속:** 데이터가 제공되는 11시부터 다음 날 0시까지 전 시간대에 걸쳐 **'폭염영향: 경고'** 상태가 지속되고 있습니다. 이는 매우 덥고 습한 날씨가 장시간 이어졌음을 의미합니다.
*   **고온 다습:** 최고 기온은 **32℃** (12시, 13시), 최고 체감기온은 **34℃** (13시)를 기록했으며, 밤에도 기온이 27℃, 체감기온이 29℃로 높은 수준을 유지했습니다. 습도는 70%에서 시작하여 밤에는 **85%**까지 상승하며 무더위와 열대야 현상에 영향을 미쳤을 것으로 보입니다.
*   **강수 없음:** 11시부터 다음 날 0시까지 강수량, 강수강도, 강수확률 모두 `null` 또는 `0%`로 기록되어, 해당 기간 동안 비는 오지 않았습니다.
*   **하늘 상태:** 11시에는 '구름 많음'으로 시작했으나, 12시부터 13시까지는 '맑음'이었습니다. 14시에 잠시 '흐림'으로 변했다가, 15시부터 다음 날 0시까지 다시 '맑음' 날씨가 이어졌습니다.
*   **바람 방향:** 오전에는 '남동풍'이 불다가 오후에 '남풍'으로 바뀌고, 밤늦게부터는 '남서풍'과 '서풍'으로 점차 서쪽으로 방향이 전환되었습니다. 바람 속도 정보는 제공되지 않았습니다.

**2. 주목할 만한 변화**

*   **기온 및 체감기온의 일 변화:**
    *   오전 11시 31℃ (체감 32℃)에서 시작하여, 오후 12시-13시에 32℃ (체감 

In [25]:
import datetime
current_time_local = datetime.datetime.now()
formatted_time= current_time_local.strftime("%Y-%m-%d %H:%M:%S")
formatted_time

'2025-08-22 13:24:56'

In [26]:
prompt = f"""다음은 시간대별 날씨 데이터입니다. 이 JSON 데이터를 분석하여 지금 외출을 한다면 우산이 필요할지 알려주세요.

**오늘 날짜 시간:** {formatted_time}

**날씨 데이터:**
```json
{json_data}
```
"""

print(prompt)

다음은 시간대별 날씨 데이터입니다. 이 JSON 데이터를 분석하여 지금 외출을 한다면 우산이 필요할지 알려주세요.

**오늘 날짜 시간:** 2025-08-22 13:24:56

**날씨 데이터:**
```json
[
  {
    "시각": "11시",
    "날씨": "구름 많음",
    "기온(℃)": 31,
    "체감기온(℃)": 32,
    "강수량(mm)": null,
    "강수강도": null,
    "강수확률(%)": null,
    "바람방향": "남동풍",
    "바람속도(m/s)": null,
    "습도(%)": 70,
    "폭염영향": "경고"
  },
  {
    "시각": "12시",
    "날씨": "맑음",
    "기온(℃)": 32,
    "체감기온(℃)": 33,
    "강수량(mm)": null,
    "강수강도": null,
    "강수확률(%)": null,
    "바람방향": "남동풍",
    "바람속도(m/s)": null,
    "습도(%)": 70,
    "폭염영향": "경고"
  },
  {
    "시각": "13시",
    "날씨": "맑음",
    "기온(℃)": 32,
    "체감기온(℃)": 34,
    "강수량(mm)": null,
    "강수강도": null,
    "강수확률(%)": null,
    "바람방향": "남동풍",
    "바람속도(m/s)": null,
    "습도(%)": 75,
    "폭염영향": "경고"
  },
  {
    "시각": "14시",
    "날씨": "흐림",
    "기온(℃)": 30,
    "체감기온(℃)": 32,
    "강수량(mm)": null,
    "강수강도": null,
    "강수확률(%)": null,
    "바람방향": "남동풍",
    "바람속도(m/s)": null,
    "습도(%)": 75,
    "폭염영향": "경고"
  },
  {
    "시각": "15시"

In [27]:
response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents=prompt,
)
print(response.text)

주어진 날씨 데이터에 따르면 **지금(2025-08-22 13:24:56)** 외출을 하신다면 **우산은 필요하지 않습니다.**

현재 시각인 13시에 해당하는 날씨 정보는 다음과 같습니다:
*   **시각:** 13시
*   **날씨:** 맑음
*   **강수량(mm):** null
*   **강수강도:** null
*   **강수확률(%):** null

13시와 바로 다음 시간인 14시에도 비 소식은 없으며, 강수량이나 강수확률도 없습니다. (14시 날씨: 흐림)

다만, **체감기온이 34℃로 매우 높고 폭염 영향이 '경고' 수준**이니, 외출 시 더위에 대한 대비를 하시는 것이 좋겠습니다.


In [28]:
prompt = f"""다음은 시간대별 날씨 데이터입니다. 이 JSON 데이터를 분석하여 지금 외출을 한다면 어떤 드레스 코디를 하는 것이 좋을지 알려주세요.

**오늘 날짜 시간:** {formatted_time}

**날씨 데이터:**
```json
{json_data}
```
"""

print(prompt)

다음은 시간대별 날씨 데이터입니다. 이 JSON 데이터를 분석하여 지금 외출을 한다면 어떤 드레스 코디를 하는 것이 좋을지 알려주세요.

**오늘 날짜 시간:** 2025-08-22 13:24:56

**날씨 데이터:**
```json
[
  {
    "시각": "11시",
    "날씨": "구름 많음",
    "기온(℃)": 31,
    "체감기온(℃)": 32,
    "강수량(mm)": null,
    "강수강도": null,
    "강수확률(%)": null,
    "바람방향": "남동풍",
    "바람속도(m/s)": null,
    "습도(%)": 70,
    "폭염영향": "경고"
  },
  {
    "시각": "12시",
    "날씨": "맑음",
    "기온(℃)": 32,
    "체감기온(℃)": 33,
    "강수량(mm)": null,
    "강수강도": null,
    "강수확률(%)": null,
    "바람방향": "남동풍",
    "바람속도(m/s)": null,
    "습도(%)": 70,
    "폭염영향": "경고"
  },
  {
    "시각": "13시",
    "날씨": "맑음",
    "기온(℃)": 32,
    "체감기온(℃)": 34,
    "강수량(mm)": null,
    "강수강도": null,
    "강수확률(%)": null,
    "바람방향": "남동풍",
    "바람속도(m/s)": null,
    "습도(%)": 75,
    "폭염영향": "경고"
  },
  {
    "시각": "14시",
    "날씨": "흐림",
    "기온(℃)": 30,
    "체감기온(℃)": 32,
    "강수량(mm)": null,
    "강수강도": null,
    "강수확률(%)": null,
    "바람방향": "남동풍",
    "바람속도(m/s)": null,
    "습도(%)": 75,
    "폭염영향": "경고"
  },
  {
   

In [29]:
response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents=prompt,
)
print(response.text)

현재 시간은 **2025년 8월 22일 13시 24분**입니다.
제공된 날씨 데이터에 따르면 13시의 날씨 정보는 다음과 같습니다:

*   **날씨:** 맑음
*   **기온(℃):** 32℃
*   **체감기온(℃):** 34℃
*   **습도(%):** 75%
*   **폭염영향:** 경고

**분석:**
현재 기온은 32℃, 체감기온은 무려 34℃로 매우 높고, 습도 또한 75%로 높아 불쾌지수가 높을 것으로 예상됩니다. 맑은 날씨에 폭염 경고까지 발령된 상황이므로, **더위를 최소화하고 햇빛으로부터 피부를 보호하는 코디**가 가장 중요합니다.

---

**지금 외출 시 추천 드레스 코디:**

1.  **의류 소재:**
    *   **가장 중요:** 땀 흡수 및 배출이 잘 되고 통기성이 좋은 **얇은 면, 리넨(마), 모달, 쿨론** 등 시원한 소재를 선택하세요.
    *   몸에 달라붙지 않고 여유 있는 핏의 옷이 좋습니다.

2.  **의류 종류:**
    *   **상의:** 얇고 넉넉한 핏의 반팔 티셔츠, 리넨 셔츠, 민소매 블라우스 등
    *   **하의:** 통풍이 잘 되는 리넨 와이드 팬츠, 롱 스커트, 시원한 소재의 반바지 (활동성에 따라)
    *   **원피스:** 얇고 하늘거리는 소재의 맥시 원피스나 롱 원피스는 온몸을 시원하게 감싸주면서 자외선 차단에도 좋습니다.
    *   **선택적 외투:** 실내외 온도차를 대비하거나 강한 햇빛으로부터 팔을 보호하고 싶다면, 매우 얇고 통기성이 좋은 긴팔 셔츠(리넨, 면)나 가디건을 준비하는 것도 좋습니다.

3.  **색상:**
    *   햇빛을 반사하여 열 흡수를 줄여주는 **밝은 색상 (흰색, 베이지, 파스텔톤)** 위주로 코디하세요. 어두운 색상은 피하는 것이 좋습니다.

4.  **신발:**
    *   통풍이 잘 되는 **샌들, 슬리퍼, 메쉬 소재의 가벼운 운동화** 등이 적합합니다. 발이 답답하지 않도록 해주세요.

5.  **필수 액세서리:**
    * 

In [30]:
formatted_time= current_time_local.strftime("%Y%m%d")

file = open(f"result_{formatted_time}.md", "w", encoding="utf8")

file.write(str(response.text))
file.close()

In [35]:
prompt = f"""다음은 시간대별 날씨 데이터입니다. 
이 JSON 데이터를 분석하여 지금 외출을 한다면 어떤 드레스 코디를 하는 것이 좋을지 알려주세요. 
결과는 현재 날씨에 어울리는 스타일로 HTML, CSS를 사용한 인포그래픽을 작성해주세요. HTML 문서 외 설명은 작성하지 마세요.

**오늘 날짜 시간:** {formatted_time}

**날씨 데이터:**
```json
{json_data}
```
"""

print(prompt)

다음은 시간대별 날씨 데이터입니다. 
이 JSON 데이터를 분석하여 지금 외출을 한다면 어떤 드레스 코디를 하는 것이 좋을지 알려주세요. 
결과는 현재 날씨에 어울리는 스타일로 HTML, CSS를 사용한 인포그래픽을 작성해주세요. HTML 문서 외 설명은 작성하지 마세요.

**오늘 날짜 시간:** 20250822

**날씨 데이터:**
```json
[
  {
    "시각": "11시",
    "날씨": "구름 많음",
    "기온(℃)": 31,
    "체감기온(℃)": 32,
    "강수량(mm)": null,
    "강수강도": null,
    "강수확률(%)": null,
    "바람방향": "남동풍",
    "바람속도(m/s)": null,
    "습도(%)": 70,
    "폭염영향": "경고"
  },
  {
    "시각": "12시",
    "날씨": "맑음",
    "기온(℃)": 32,
    "체감기온(℃)": 33,
    "강수량(mm)": null,
    "강수강도": null,
    "강수확률(%)": null,
    "바람방향": "남동풍",
    "바람속도(m/s)": null,
    "습도(%)": 70,
    "폭염영향": "경고"
  },
  {
    "시각": "13시",
    "날씨": "맑음",
    "기온(℃)": 32,
    "체감기온(℃)": 34,
    "강수량(mm)": null,
    "강수강도": null,
    "강수확률(%)": null,
    "바람방향": "남동풍",
    "바람속도(m/s)": null,
    "습도(%)": 75,
    "폭염영향": "경고"
  },
  {
    "시각": "14시",
    "날씨": "흐림",
    "기온(℃)": 30,
    "체감기온(℃)": 32,
    "강수량(mm)": null,
    "강수강도": null,
    "강수확률(%)": null,
    "바람방향": "남동풍",
   

In [36]:
response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents=prompt,
)
print(response.text)

```html
<!DOCTYPE html>
<html lang="ko">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>오늘의 드레스 코드: 폭염 경고</title>
    <link href="https://fonts.googleapis.com/css2?family=Noto+Sans+KR:wght@300;400;700&display=swap" rel="stylesheet">
    <style>
        :root {
            --primary-bg: #fff;
            --secondary-bg: #f9f9f9;
            --accent-color: #ff9900; /* Warm orange */
            --text-color: #333;
            --light-text: #666;
            --border-color: #eee;
            --shadow-color: rgba(0, 0, 0, 0.1);
            --warning-color: #dc3545; /* Red for warning */
            --recommendation-color: #4CAF50; /* Green for recommendations */
            --tip-bg-color: #e0f2f7; /* Light blue for tips */
            --tip-border-color: #03a9f4; /* Darker blue for tip border */
            --tip-text-color: #01579b; /* Even darker blue for tip text */
            --temp-color: #d35400; /* Darker or

In [37]:
result = response.text
result = result.replace("```html","").replace("```","")
print(result)


<!DOCTYPE html>
<html lang="ko">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>오늘의 드레스 코드: 폭염 경고</title>
    <link href="https://fonts.googleapis.com/css2?family=Noto+Sans+KR:wght@300;400;700&display=swap" rel="stylesheet">
    <style>
        :root {
            --primary-bg: #fff;
            --secondary-bg: #f9f9f9;
            --accent-color: #ff9900; /* Warm orange */
            --text-color: #333;
            --light-text: #666;
            --border-color: #eee;
            --shadow-color: rgba(0, 0, 0, 0.1);
            --warning-color: #dc3545; /* Red for warning */
            --recommendation-color: #4CAF50; /* Green for recommendations */
            --tip-bg-color: #e0f2f7; /* Light blue for tips */
            --tip-border-color: #03a9f4; /* Darker blue for tip border */
            --tip-text-color: #01579b; /* Even darker blue for tip text */
            --temp-color: #d35400; /* Darker orange fo

In [39]:
formatted_time= current_time_local.strftime("%Y%m%d")

file = open(f"result_{formatted_time}.html", "w", encoding="utf8")

file.write(str(result))
file.close()